# 学習の見守り

裏で動いている学習（`scripts/train_*.sh`）の進み具合を 1 分ごとに表示し、学習が終わるまで待つ。
標準のカーネル（Python 3）でも動く。`RUN` を学習の実行名にして Run All Cells。

- 表示は 1 分ごとに最新の 1 行（`step:...`）に置き換わる。学習が終わるとセルの実行も終わる。
- このセルが動いている間はノートブックが「実行中」になるので、SageMaker の「操作がないとスペースを自動停止する」
  設定があっても止まりにくくなる。ブラウザを閉じてもセルは動き続ける。
- 学習そのものはこのノートブックとは別に動いているので、このセルを止めても（■ ボタン）学習は止まらない。


In [ ]:
RUN = "smolvla_pot_multi_v1"  # 見守る学習の実行名（scripts/train_*.sh の 2 番目の引数）

import subprocess
import time
from pathlib import Path

from IPython.display import clear_output

repo = Path.cwd().resolve()
if repo.name == "notebooks":
    repo = repo.parent
logs = sorted((repo / "logs").glob(f"train_{RUN}_*.log"))
if not logs:
    raise FileNotFoundError(f"logs/train_{RUN}_*.log がありません。学習を始めてから実行してください。")
log = logs[-1]

while True:
    # lerobot-train のコマンドラインに --output_dir=outputs/train/<RUN> があるプロセスを探す
    pattern = f"lerobot-train.*outputs/train/{RUN}( |$)"
    running = subprocess.run(["pgrep", "-f", pattern], capture_output=True).returncode == 0
    steps = [line for line in log.read_text(errors="ignore").splitlines() if "step:" in line]
    clear_output(wait=True)
    print(f"{time.strftime('%H:%M:%S')}  {log.name}  {'学習中' if running else '学習は終わっています'}")
    if steps:
        print(steps[-1].split("\r")[-1][-300:])
    if not running:
        break
    time.sleep(60)
